# 02 - Preprocessing and training the autoencoder
**Key idea:** train ONLY on normal traffic. Anything the model can't rebuild well is suspicious.

In [1]:
import sys; sys.path.append("..")
import json, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from src.features import COLS, build_matrix
from src.model import Autoencoder

torch.manual_seed(42); np.random.seed(42)
train_df = pd.read_csv("../data/raw/KDDTrain+.txt", names=COLS)
test_df = pd.read_csv("../data/raw/KDDTest+.txt", names=COLS)
train_df["is_attack"] = (train_df["label"] != "normal").astype(int)
test_df["is_attack"] = (test_df["label"] != "normal").astype(int)

## Column list ("vocabulary") from the training file, built once

In [2]:
feature_columns = list(build_matrix(train_df).columns)
print(len(feature_columns), "columns after one-hot encoding")

121 columns after one-hot encoding


## Split: only NORMAL rows are used for training
- `train_normal` 80%: trains the model and fits the scaler
- `val_normal` 20%: early stopping and threshold choice
- test file: final evaluation only

In [3]:
normal_df = train_df[train_df["is_attack"] == 0]
train_n, val_n = train_test_split(normal_df, test_size=0.2, random_state=42)
print("train_normal:", train_n.shape, "| val_normal:", val_n.shape, "| test:", test_df.shape)

train_normal: (53874, 44) | val_normal: (13469, 44) | test: (22544, 44)


## Build matrices and scale (scaler fitted on normal training data only)

In [4]:
X_train_raw = build_matrix(train_n, feature_columns)
X_val_raw = build_matrix(val_n, feature_columns)
X_test_raw = build_matrix(test_df, feature_columns)

scaler = MinMaxScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)
y_test = test_df["is_attack"].values
print(X_train.shape, X_val.shape, X_test.shape, y_test.shape)

(53874, 121) (13469, 121) (22544, 121) (22544,)


## Sanity checks (never skip)

In [5]:
for name, arr in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(name, "NaN:", np.isnan(arr).sum(), "inf:", np.isinf(arr).sum())
print("train min/max:", X_train.min(), X_train.max())
print("test min/max:", X_test.min(), X_test.max())
# The API scores ONE row at a time: it must match the batch result exactly
one = scaler.transform(build_matrix(test_df.iloc[[0]], feature_columns)).astype(np.float32)
print("single-row == batch row:", np.allclose(one[0], X_test[0]))

train NaN: 0 inf: 0
val NaN: 0 inf: 0
test NaN: 0 inf: 0
train min/max: 0.0 1.0
test min/max: 0.0 3.0
single-row == batch row: True


## Save arrays and the scaler

In [6]:
np.save("../data/processed/X_train_normal.npy", X_train)
np.save("../data/processed/X_val_normal.npy", X_val)
np.save("../data/processed/X_test.npy", X_test)
np.save("../data/processed/y_test.npy", y_test)
joblib.dump(scaler, "../artifacts/scaler.joblib")

['../artifacts/scaler.joblib']

## Train the autoencoder
Loop per batch: `zero_grad -> forward -> loss -> backward -> step`.
Early stopping: keep the weights from the epoch with the lowest validation loss.

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
BOTTLENECK, LR, BATCH, MAX_EPOCHS, PATIENCE = 16, 1e-3, 256, 60, 6
input_dim = X_train.shape[1]
model = Autoencoder(input_dim, BOTTLENECK).to(device)
loss_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=BATCH, shuffle=True)
Xv = torch.from_numpy(X_val).to(device)

best_val, best_state, bad, history = float("inf"), None, 0, []
for epoch in range(1, MAX_EPOCHS + 1):
    model.train(); total = 0.0
    for (xb,) in train_loader:
        xb = xb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), xb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    tr = total / len(X_train)
    model.eval()
    with torch.no_grad():
        va = loss_fn(model(Xv), Xv).item()
    history.append((tr, va))
    if va < best_val - 1e-7:
        best_val, bad = va, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
    if epoch % 5 == 0 or bad == 0:
        print(f"epoch {epoch:3d}  train {tr:.6f}  val {va:.6f}")
    if bad >= PATIENCE:
        print("early stopping at epoch", epoch); break

model.load_state_dict(best_state)
torch.save(best_state, "../artifacts/autoencoder.pth")
print("best val loss:", round(best_val, 6))

epoch   1  train 0.018690  val 0.007086


epoch   2  train 0.004300  val 0.003295


epoch   3  train 0.002611  val 0.002093


epoch   4  train 0.001788  val 0.001656


epoch   5  train 0.001488  val 0.001438


...

epoch  54  train 0.000364  val 0.000410


epoch  55  train 0.000364  val 0.000416


epoch  56  train 0.000359  val 0.000408


epoch  57  train 0.000357  val 0.000406


epoch  60  train 0.000350  val 0.000407
best val loss: 0.000406


In [8]:
import os; os.makedirs("../figures", exist_ok=True)
h = np.array(history)
plt.plot(h[:, 0], label="train"); plt.plot(h[:, 1], label="validation")
plt.xlabel("epoch"); plt.ylabel("MSE loss"); plt.legend(); plt.title("Training curves")
plt.savefig("../figures/loss_curves.png", dpi=120, bbox_inches="tight"); plt.show()

## Save metadata (the API needs the column list and the layer sizes)

In [9]:
with open("../artifacts/metadata.json", "w") as f:
    json.dump({"feature_columns": feature_columns, "input_dim": int(input_dim),
               "bottleneck": BOTTLENECK}, f, indent=2)
print("saved")

saved
